In [38]:
#Imporitng main libraries
import pandas as pd
import numpy as np
from pathlib import Path
import os


In [39]:
# loading the data

base_path = Path(os.getcwd())

input_path  = base_path / r"..\..\Data\Raw\df_bologna_full.parquet"

df = pd.read_parquet(input_path)

In [40]:
df.dtypes

time             datetime64[ns]
latitude                float64
longitude               float64
t2m                     float32
SSRD_Wm2                float32
CLEAR_SKY_GHI           float64
GHI                     float64
dtype: object

## Data Engineering

In [41]:
#Convert the temperature from Kelvin to Celsius
df['t2m'] = df['t2m'] - 273.15

# Create the GHI index for each hour as the ratio of actual GHI to clear sky GHI
df['GHI_index'] = df['GHI'] / df['CLEAR_SKY_GHI']

In [42]:
# find rows with any missing values
missing_rows = df[df.isna().any(axis=1)]
print(f"Rows with missing values: {len(missing_rows)}")
missing_rows

# or if you only care about GHI
ghi_missing = df[df['GHI'].isna()]
print(f"GHI-missing rows: {len(ghi_missing)}")
ghi_missing

Rows with missing values: 84105
GHI-missing rows: 53


,time,latitude,longitude,t2m,SSRD_Wm2,CLEAR_SKY_GHI,GHI,GHI_index
246,2005-01-11 07:00:00,45.5,11.25,0.465149,0.017778,0.152133,NaN,NaN
247,2005-01-11 08:00:00,45.5,11.25,1.397980,44.337776,56.452446,NaN,NaN
248,2005-01-11 09:00:00,45.5,11.25,3.015594,148.800003,187.463211,NaN,NaN
249,2005-01-11 10:00:00,45.5,11.25,5.590363,251.822220,302.815887,NaN,NaN
250,2005-01-11 11:00:00,45.5,11.25,6.629700,335.662231,375.320709,NaN,NaN
251,2005-01-11 12:00:00,45.5,11.25,6.859314,304.444458,395.509888,NaN,NaN
252,2005-01-11 13:00:00,45.5,11.25,7.277710,308.764435,361.091614,NaN,NaN
253,2005-01-11 14:00:00,45.5,11.25,7.609955,250.008896,276.060822,NaN,NaN
254,2005-01-11 15:00:00,45.5,11.25,7.172729,140.160004,153.207199,NaN,NaN
255,2005-01-11 16:00:00,45.5,11.25,6.764374,25.777779,31.448774,NaN,NaN


# Data cleaning

## Filling missing values

In [43]:
# 1. Setup: Ensure datetime and sorted
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time')

# Create a temporary date column for grouping
df['temp_date'] = df['time'].dt.date

# RULE 1: Nighttime is 0
df.loc[(df['GHI'].isna()) & (df['CLEAR_SKY_GHI'] == 0), 'GHI'] = 0

# --- RULE 2: Fill based on Daily Mean Ratio ---

# Calculate the hourly ratio (Clearness Index) where data exists
df['ratio'] = df['GHI'] / df['CLEAR_SKY_GHI']
# Replace infinite values (from dividing by zero) with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Calculate the mean ratio per day (ignoring NaNs)
daily_mean_ratio = df.groupby('temp_date')['ratio'].transform('mean')

# Apply the ratio fill: GHI = CLEAR_SKY_GHI * daily_mean_ratio
df['GHI'] = df['GHI'].fillna(df['CLEAR_SKY_GHI'] * daily_mean_ratio)

# --- RULE 3: Fill full-day gaps with Clear Sky values ---

# If daily_mean_ratio was NaN (entire day missing), use Clear Sky directly
df['GHI'] = df['GHI'].fillna(df['CLEAR_SKY_GHI'])

# 4. Final Cleanup: Remove helper columns
df = df.drop(columns=['temp_date', 'ratio'])

print("Missing values remaining:", df['GHI'].isna().sum())

Missing values remaining: 0


# Saving the cleaned dataframe

In [44]:
#Save the dataframe

OUTPUT_CLEAN  = base_path / r"..\..\Data\Cleaned\df_bologna_cleaned.parquet"

df.reset_index().to_parquet(OUTPUT_CLEAN, index=False)
print(f"\n[SAVED] {OUTPUT_CLEAN}")
print(f"  Final shape: {df.shape}")


[SAVED] c:\Users\LucasMonero\Documents\data projects\Master Thesis\Project\Code\Transformation\..\..\Data\Cleaned\df_bologna_cleaned.parquet
  Final shape: (184079, 8)
